# 视频 → 多人检测/跟踪 → 每人 JSON + 原始 NPZ
默认 **GPU / 多人模式**。单个视频只解码一次：torchvision Faster R-CNN 检测 person → 预测bbox IoU跨帧关联 → 每人SAM推理 → 独立JSON1.0/NPZ。原Blender driver每次读取一个人的JSON即可。
这不是可靠身份识别系统：交叉/长遮挡可能换ID，超过TTL重新进入会分配新ID。正式运行先测试短片并人工核对。
以下各阶段有独立可执行代码：官方依赖补缺、git同步、SAM模型下载、检测器下载、GPU/CPU资源检查、多人/单人推理。外部安装/下载默认关闭，按需要开启；已有PyTorch/torchvision/CUDA绝不重新安装。

In [ ]:
from pathlib import Path
import os, sys
# 在 extractor 目录打开notebook；否则修改此路径
EXTRACTOR_DIR = Path.cwd().resolve()
sys.path.insert(0, str(EXTRACTOR_DIR))
VIDEO = Path("input.mp4").resolve()
DEVICE = "cuda"  # 默认GPU；CPU切换见下面默认注释的补丁单元
MULTI_PERSON = True
OUTPUT_DIR = Path("motion_multi")  # 多人：必须是新目录
OUTPUT_JSON = Path("motion_single.json")  # 单人模式使用
SAM_REPO = Path("../sam-3d-body").resolve()
DINOV3_REPO = Path("../dinov3").resolve()
CHECKPOINT_DIR = Path("../checkpoints").resolve()
CHECKPOINT = CHECKPOINT_DIR / "model.ckpt"
MHR_MODEL = CHECKPOINT_DIR / "assets/mhr_model.pt"
DETECTOR_WEIGHTS = Path("../detector/fasterrcnn_resnet50_fpn_v2_coco.pth").resolve()
STRIDE = 1
MAX_FRAMES = 30  # 先测30个采样帧；完整视频改None
START_SECONDS = 0.0
END_SECONDS = None
DETECTOR_THRESHOLD = 0.7
MAX_PEOPLE = 20  # 超限只保留高分检测，其余可能missing
TRACK_IOU_THRESHOLD = 0.25
MAX_MISSING_SECONDS = 1.0  # 必须大于正常采样帧间隔；按真实秒而非帧数
SAVE_NPZ = True
NPZ_COMPRESSED = True
INFERENCE_TYPE = "body"
CONTINUE_ON_POSE_ERROR = False
SINGLE_BBOX = None  # 单人模式: [x1,y1,x2,y2] 或全图None
RUN_INSTALL = False
RUN_GIT_SYNC = False
RUN_MODEL_DOWNLOAD = False
RUN_DETECTOR_DOWNLOAD = False
MODEL_SOURCE = "modelscope"  # 第三方公开镜像；可选"huggingface"官方源
MODEL_REVISION = "master" if MODEL_SOURCE == "modelscope" else "main"
MODEL_TERMS_CONFIRMED = False
DETECTOR_TERMS_CONFIRMED = False
os.environ["SAM3D_DINOV3_REPO"] = str(DINOV3_REPO)


## 1. 官方 INSTALL.md Python依赖：仅安全补缺
已核对[固定SAM版本INSTALL.md](https://github.com/facebookresearch/sam-3d-body/blob/b5c765a0d89d789985e186d396315e7590887b94/INSTALL.md)第3节全部35项，另加视频PyAV、YAML与ModelScope客户端。安装目标是当前kernel；当前环境须已有torch/torchvision及所需CUDA，缺少就停止，绝不自动补装。
开关RUN_INSTALL=True才执行。下方打印具体依赖后：冻结所有已装版本，wheel-only dry-run检查计划，拒绝任何已有包替换/重装和新的torch/CUDA家族包，按原wheel URL+SHA256仅补缺，最后pip check。OpenCV不同provider共用cv2目录时也拒绝。没有wheel/版本冲突会停止，不自动绕过保护或源码构建。--no-deps并不假装依赖完整：闭包先解析，失败仍需处理；--no-build-isolation不安装额外编译环境。Detectron2/MoGe/SAM3不需要，检测器直接用已有torchvision。
新包安装可能部分完成，建议专用已有环境，不要并发安装其他包；实际import/ABI仍需检查。安装后重启kernel再继续。

In [ ]:
from setup_runtime import OFFICIAL_DEPENDENCIES, PIPELINE_EXTRAS, install_dependencies
print("官方Python依赖:", " ".join(OFFICIAL_DEPENDENCIES))
print("视频管线补充:", PIPELINE_EXTRAS)
if RUN_INSTALL:
    install_dependencies()  # 当前sys.executable；不会安装torch/torchvision/CUDA
else:
    print("跳过安装。需要时开启RUN_INSTALL并单独运行本单元。")


## 2. git同步：固定官方源码，不覆盖现有checkout
需要系统git。首次clone后checkout固定commit；已有目录只验证commit/origin/已知patch，不执行pull/reset，不覆盖用户修改。“同步”指确保这两个固定版本，不是追逐远端HEAD。
默认GPU路径只应用本地DINOv3和安全checkpoint读取补丁 `gpu_local_dinov3.patch`，不会做CPU device改写。因此DINOv3不会偷偷从GitHub HEAD加载，也不需要Detectron2。旧完整CPU补丁仍兼容GPU。

In [ ]:
from setup_runtime import SAM_URL, SAM_COMMIT, DINO_URL, DINO_COMMIT, ensure_repo, setup_sources
print("SAM:", SAM_URL, SAM_COMMIT)
print("DINOv3:", DINO_URL, DINO_COMMIT)
if RUN_GIT_SYNC:
    # 等效动作: git clone --no-checkout <官方repo>; git checkout --detach <固定commit>
    ensure_repo(SAM_REPO, SAM_URL, SAM_COMMIT)
    ensure_repo(DINOV3_REPO, DINO_URL, DINO_COMMIT)
    # 仅已知GPU安全/本地DINO patch，拒绝未知修改
    setup_sources(SAM_REPO, DINOV3_REPO)
else:
    print("跳过git同步。已有正确源码可复用，否则开启RUN_GIT_SYNC。")


### 可选CPU：代码默认注释
GPU用户不要运行此段。CPU需先完成上面的本地DINO源码设置，再取消下面三行注释，应用显式device兼容补丁并切换DEVICE。该补丁和GPU的本地DINO补丁合起来等价于旧版cpu_local_dinov3.patch。不会安装CPU版torch，不会替换CUDA环境。改源码或设备后重启kernel，重新配置DEVICE='cpu'；check_runtime会确认CPU补丁确实存在。

In [ ]:
# from setup_runtime import apply_cpu_compatibility_patch
# apply_cpu_compatibility_patch(SAM_REPO)
# DEVICE = "cpu"


## 3. SAM模型下载（明确来源、文件与目录）
默认ModelScope `facebook/sam-3d-body-dinov3`，此前公开可下载，不要求Hugging Face访问申请；这是第三方镜像，仍有模型和来源许可。官方HF同名源可能gated，需先获得访问权限并在终端安全登录；不要把token放进notebook。
仅下载 model.ckpt、model_config.yaml、assets/mhr_model.pt 到上方CHECKPOINT_DIR。已有目录只校验、不覆盖；不完整请选择新目录。约2GB模型，下载前需至少8GiB临时/缓存空间；含依赖建议15–20GB以上。SHA256默认与此前已使用文件一致，但不是官方认证；变化会停下，不能盲目清空校验。HF解析记录commit，ModelScope记录revision和逐文件hash，不伪造不可变commit。

In [ ]:
from setup_runtime import MODEL_ID, MODEL_FILES, KNOWN_HASHES, download_assets
print("源/模型/版本:", MODEL_SOURCE, MODEL_ID, MODEL_REVISION)
print("下载文件:", MODEL_FILES, "目标:", CHECKPOINT_DIR)
if RUN_MODEL_DOWNLOAD:
    sam_assets = download_assets(CHECKPOINT_DIR, source=MODEL_SOURCE, revision=MODEL_REVISION,
                                expected_hashes=KNOWN_HASHES, terms_confirmed=MODEL_TERMS_CONFIRMED)
    print(sam_assets)
else:
    print("跳过SAM下载。阅读来源许可后设置MODEL_TERMS_CONFIRMED和RUN_MODEL_DOWNLOAD。")


## 4. 多人检测器下载：torchvision官方Faster R-CNN
只用COCO person类（label=1）。下载地址来自当前已装torchvision的 `FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1`，只允许download.pytorch.org官方HTTPS源并检查URL hash前缀；不安装torchvision，不会隐式下载backbone。模型文件约百余MB，建议至少1GiB可用空间。
[torchvision代码许可证](https://github.com/pytorch/vision/blob/main/LICENSE)与[预训练权重/数据的条款](https://github.com/pytorch/vision#pre-trained-model-license)不是同一回事；阅读后确认，不宣称权重一概无限制商业使用。此选型不是额外人物身份识别模型。单人模式可跳过。

In [ ]:
if MULTI_PERSON:
    from torchvision.models.detection import FasterRCNN_ResNet50_FPN_V2_Weights
    from setup_runtime import download_detector_weights
    print("检测器模型: FasterRCNN_ResNet50_FPN_V2 / COCO_V1")
    print("官方权重:", FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1.url)
    print("本地目标:", DETECTOR_WEIGHTS)
    if RUN_DETECTOR_DOWNLOAD:
        download_detector_weights(DETECTOR_WEIGHTS, terms_confirmed=DETECTOR_TERMS_CONFIRMED)
    else:
        print("跳过检测器下载，加载前必须有本地权重。")


## 5. 本地资源/导入检查与模型加载
下面不下载模型；检查SAM/DINO版本与GPU/CPU对应patch、模型hash/config、导入路径与CUDA可用性。GPU内存同时承载检测器与SAM，不保证任何显卡都能运行；OOM应缩短/降低负载或选合适硬件，不能把失败当无人。首次hash读取较慢。

In [ ]:
from setup_runtime import check_runtime
from extract_video import load_estimator, preflight, preflight_archive
import torch
if not VIDEO.is_file():
    raise FileNotFoundError(VIDEO)
validated_assets = check_runtime(SAM_REPO, DINOV3_REPO, CHECKPOINT_DIR, target=DEVICE, expected_hashes=KNOWN_HASHES)
if MULTI_PERSON:
    if OUTPUT_DIR.exists():
        raise FileExistsError("请使用新OUTPUT_DIR，防止多次运行混合")
    if not DETECTOR_WEIGHTS.is_file():
        raise FileNotFoundError("先运行检测器下载单元")
else:
    preflight(VIDEO, OUTPUT_JSON)
    if SAVE_NPZ:
        preflight_archive(OUTPUT_JSON)
estimator, runtime_metadata = load_estimator(SAM_REPO, CHECKPOINT, MHR_MODEL, DEVICE)
runtime_metadata["validated_assets"] = validated_assets
if MULTI_PERSON:
    from multi_person import TorchvisionPersonDetector
    detector = TorchvisionPersonDetector(DETECTOR_WEIGHTS, device=DEVICE,
                                        score_threshold=DETECTOR_THRESHOLD, max_people=MAX_PEOPLE)
print(runtime_metadata)


## 6. 视频多人推理 / 保留单人兼容模式
检测与跟踪发生在采样帧，时间来自真实PTS；增大STRIDE会降低身份关联能力。MAX_MISSING_SECONDS须大于正常采样间隔（如STRIDE/source FPS），否则连续可见者也可能被判过期。算法为确定性的constant-velocity bbox + greedy IoU匹配，不含外观重识别；短漏检保留ID，超TTL回归是新ID，交叉/遮挡仍可能换ID。
每人文件夹有motion.json（原schema1.0）和raw_npz；不把预测框变成假姿态。缺失记录missing，逐人pose失败可按开关记录error；检测器异常始终停止。无人时完成manifest但无人物JSON；某人全失败仅留诊断。原始frame_index/time_seconds不会因某人晚出现而归零。

In [ ]:
with torch.inference_mode():
    if MULTI_PERSON:
        from multi_person import extract_multi
        result = extract_multi(VIDEO, OUTPUT_DIR, estimator, detector,
            stride=STRIDE, max_frames=MAX_FRAMES,
            start_seconds=START_SECONDS, end_seconds=END_SECONDS,
            iou_threshold=TRACK_IOU_THRESHOLD, max_missing_seconds=MAX_MISSING_SECONDS,
            inference_type=INFERENCE_TYPE, save_npz=SAVE_NPZ, npz_compressed=NPZ_COMPRESSED,
            continue_on_error=CONTINUE_ON_POSE_ERROR, metadata=runtime_metadata)
        print("状态:", result["status"], "采样帧:", result["sampled_frames"])
        for person in result["tracks"]:
            print("track", person["track_id"], person["status"], person["json_file"])
    else:
        from extract_video import extract
        result = extract(VIDEO, OUTPUT_JSON, estimator, stride=STRIDE, max_frames=MAX_FRAMES,
            start_seconds=START_SECONDS, end_seconds=END_SECONDS, bbox=SINGLE_BBOX,
            inference_type=INFERENCE_TYPE, metadata=runtime_metadata,
            save_npz=SAVE_NPZ, npz_compressed=NPZ_COMPRESSED,
            continue_on_error=CONTINUE_ON_POSE_ERROR)
        print("单人JSON:", OUTPUT_JSON.resolve())


## 7. 交给原Blender工程 / 查看原始数值
多人例：motion_multi/track_0001/motion.json、track_0002/motion.json分别传给原Blender driver，每次得到一份独立角色动画。当前不会自动把多个人物放到一个共享场景，也不会恢复他们彼此的世界位置或原片摄影机。
manifest.json必须complete=true才表示本次导出完成；中断可能已写入各track的partial NPZ，但根manifest未汇总，不可当作完整结果。samples.jsonl记录每个采样帧的观察/活跃/过期ID。每轨raw_npz的manifest和frames.jsonl保存track_id、源帧号和时间。
NPZ逐帧写盘；每轨JSON关键点仍保留内存，长视频/多人会线性增加内存和存储，请分段。只用allow_pickle=False加载可信NPZ。

In [ ]:
import json
import numpy as np
if MULTI_PERSON:
    manifest_path = OUTPUT_DIR / "manifest.json"
    completed = json.loads(manifest_path.read_text(encoding="utf-8"))
    assert completed["complete"]
    print("有效人物JSON:", [p["json_file"] for p in completed["tracks"] if p["json_file"]])
    if SAVE_NPZ:
        candidates = OUTPUT_DIR.glob("track_*/raw_npz/frame_*.npz")
        first = next(candidates, None)
        if first is not None:
            with np.load(first, allow_pickle=False) as raw:
                print(first, {k: raw[k].shape for k in raw.files})
